In [ ]:
import re
from collections import Counter
import pandas as pd

# 1) define your targets (and allow optional plural 's')
targets = ['garden', 'farm', 'allotment', 'school', 'orchard', 'plot', 'park']

# 2) initialize a Counter for each
preceding = {t: Counter() for t in targets}

# 3) for each document, find all matches of "<word> <target>" or "<word> <target>s"
for text in df['corpus'].fillna(''):
    for t in targets:
        # regex: capture the word before, allow optional plural
        pattern = rf'\b(\w+)\s+{re.escape(t)}s?\b'
        for match in re.findall(pattern, text):
            preceding[t].update([match.lower()])

# 4) show the top 10 for each target
for t in targets:
    print(f"\nTop words before '{t}':")
    for word, cnt in preceding[t].most_common(10):
        print(f"  {word:15} {cnt}")


Top words before 'garden':
  community       160
  energy          26
  kitchen         17
  edible          16
  school          10
  herb            9
  forest          9
  network         8
  sensory         7
  one             7

Top words before 'farm':
  city            6
  urban           3
  clitterhouse    3
  forest          3
  community       2
  loughborough    2
  mushroom        2
  dock            2
  patchwork       2
  broadwater      2

Top words before 'allotment':
  community       15
  garden          10
  road            4
  park            3
  lane            3
  council         2
  event           2
  plot            2
  street          2
  raised          2

Top words before 'school':
  primary         19
  forest          5
  garden          4
  local           3
  child           3
  field           3
  woodmansterne   2
  within          2
  resident        2
  heart           1

Top words before 'orchard':
  community       12
  garden          4
  fruit           3
  urban           3
  westmacott      2
  henley          2
  communal        2
  meadow          2
  tunnel          1
  apple           1

Top words before 'plot':
  allotment       10
  community       5
  individual      4
  growing         3
  vegetable       3
  new             3
  acre            2
  restore         1
  cultivating     1
  mitcham         1

Top words before 'park':
  garden          4
  gunnersbury     4
  hill            3
  public          3
  maryon          3
  car             3
  memorial        2
  area            2
  lewisham        2
  small           2

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# 1) instantiate a vectorizer that only looks at 2-word sequences
bigram_vect = CountVectorizer(
    ngram_range=(2, 2),
    stop_words='english'   # drop common English stop-words
)

# 2) fit it on your corpus
X2 = bigram_vect.fit_transform(df['corpus'].fillna(''))

# 3) sum up counts for each bigram
counts = X2.sum(axis=0).A1
bigrams = bigram_vect.get_feature_names_out()

# 4) assemble into a DataFrame and sort by frequency
bigram_freq = pd.DataFrame({
    'bigram': bigrams,
    'count': counts
}).sort_values('count', ascending=False)

# 5) view the top 50 most common bigrams
print(bigram_freq.head(50))

bigram  count
1412     community garden    160
2701         food growing     75
6705           raised bed     74
3712        growing space     46
3010     garden community     39
2913           fruit tree     31
2205        energy garden     26
4807      local community     25
2916      fruit vegetable     23
4824       local resident     23
6392       primary school     19
3463          green space     18
4520       kitchen garden     17
1417      community group     17
3697      growing project     17
1407       community food     16
2093        edible garden     16
3651         growing food     16
1379  community allotment     15
1387     community centre     15
3227           garden run     14
3217        garden raised     14
1420    community growing     14
5781       park community     14
3069        garden garden     14
241        allotment site     14
3574            grow food     14
3244         garden small     13
5168        mental health     13
3611       grow vegetable     13
5664               org uk     12
1449    community orchard     12
8614      vegetable fruit     11
3352    gardening session     11
8620       vegetable herb     11
227        allotment plot     10
3138        garden london     10
4821         local people     10
2971     garden allotment     10
8965         wide variety     10
3627         growing area     10
3535          group local     10
7160        school garden     10
3615            grow wide      9
5586             open day      9
3968          herb garden      9
7710         space people      9
1475      community space      9
2771        forest garden      9
2915            fruit veg      9

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd

texts = df["corpus"].fillna("")

vec = TfidfVectorizer(max_df=0.9, min_df=3, ngram_range=(1,2), stop_words="english")
X = vec.fit_transform(texts)

# Try several k and pick by silhouette
ks = range(3, 12)
sils = []
for k in ks:
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    labs = km.fit_predict(X)
    sils.append(silhouette_score(X, labs))
best_k = ks[int(np.argmax(sils))]
print("Best k:", best_k, "silhouette:", max(sils))

km = KMeans(n_clusters=best_k, n_init="auto", random_state=42)
labels = km.fit_predict(X)
df["cluster_km"] = labels

# Inspect: top terms per cluster
terms = vec.get_feature_names_out()
def top_terms(cl, n=10):
    cent = km.cluster_centers_[cl]
    idx = np.argsort(cent)[-n:][::-1]
    return [terms[i] for i in idx]

for c in range(best_k):
    print(f"\nCluster {c}: {', '.join(top_terms(c, 12))}")
    print(df.loc[df.cluster_km==c, "space_name"].head(5).to_list())

Best k: 5 silhouette: 0.014059937747918985

Cluster 0: school, garden, primary, child, primary school, project, area, school garden, planter, plant, edible, space
['Applegarth Academy', 'Woodmansterne School Farm', 'Rockmount Primary School', 'Benedict House Prep School', 'Furzedown Project&#39;s Furzedown Farmers&#39; Veg Plot at Streatham Park Bowling Club']

Cluster 1: food, community, growing, food growing, garden, space, project, community garden, people, local, nature, volunteer
['GOOD FOOD MATTERS', 'EcoLocal', 'May Project Food Growing project', 'Parkfields Community Garden', 'The Plot']

Cluster 2: garden, fruit, bed, tree, orchard, raised, park, raised bed, growing, area, community, space
['Science Garden', 'Park Hill Park Community Garden', 'Mitcham Community Orchard and Gardens', 'Invisible Palace', 'Norwood Park community gardens']

Cluster 3: allotment, plot, allotment site, site, allotment plot, community allotment, allotment allotment, community, garden allotment, acton, year, road
['Southlands Road Allotments', 'Tamworth Farm Allotments', 'Plot 1 - North Mitcham Plot Owners Association (NMPOA)', 'Grow Lewisham', 'Firhill South Allotments']

Cluster 4: garden, community, community garden, grow, garden community, space, farm, london, group, estate, open, green
['Sutton Community Farm', 'MHA The Wilderness', 'Fishponds Community Garden', 'Phipps Bridge Community Garden', 'Deen City Farm']

In [ ]:

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt


# Use the same X from TF-IDF
Z = linkage(X.toarray(), method="ward")  # for small/medium N; for large N, sample
plt.figure(figsize=(10,5)); dendrogram(Z, no_labels=True, color_threshold=None); plt.show()

# Pick a cluster count and fit
agg = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
df["cluster_hc"] = agg.fit_predict(X.toarray())


In [ ]:
df = dd.copy()

df["parts"] = df["Variable"].astype(str).str.split("-")

# last chunk
df["last1"] = df["parts"].str[-1]

df["last1"].value_counts()

In [ ]:
def map_exp_rec(code: str) -> str:
    if not isinstance(code, str):
        return "Other"
    code = code.lower()
    if code.startswith("exp"):
        return "Expenditure"
    elif code.startswith("rec"):
        return "Receipts"
    else:
        return "Other"

dd["exp_rec"] = dd["last1"].apply(map_exp_rec)


In [ ]:

# Helper that works whether columns are named or not
def get_by_variable(df: pd.DataFrame, label: str) -> pd.Series:
    if isinstance(df.columns, pd.MultiIndex):
        s = df.loc[:, df.columns.get_level_values(-1) == label]
        return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s
    return df[label]

# 1) Identifiers
la_name = get_by_variable(rev, "LA_name").astype(str).str.strip()
year    = get_by_variable(rev, "year_ending").astype(str).str.strip()

# 2) Keep ONLY measure columns (drop identifiers before aggregating)
id_vars = {"LA_name","year_ending"}
last_level = -1  # last level holds the raw variable names
measure_mask = ~rev.columns.get_level_values(last_level).isin(id_vars)
rev_meas = rev.loc[:, measure_mask]

# 3) Collapse columns by category × subcategory (fast and vectorized)
#    -> result has columns indexed by (category, subcategory)
rev_by_cat = rev_meas.groupby(axis=1, level=["category","subcategory"]).sum()

# 4) Add row identifiers, then aggregate by borough × year
rev_by_cat = rev_by_cat.assign(LA_name=la_name.values, year=year.values)

rev_agg = (
    rev_by_cat
    .groupby(["LA_name","year"], as_index=False)
    .sum(numeric_only=True)
    # at this point columns are the (category, subcategory) MultiIndex, plus the two group keys if you keep them
)

# Optional: keep a tidy “wide” table indexed by borough × year, with (category, subcategory) columns
rev_wide = (
    rev_by_cat
    .groupby(["LA_name","year"])
    .sum(numeric_only=True)
)


In [ ]:

# Helper that works whether columns are named or not
def get_by_variable(df: pd.DataFrame, label: str) -> pd.Series:
    if isinstance(df.columns, pd.MultiIndex):
        s = df.loc[:, df.columns.get_level_values(-1) == label]
        return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s
    return df[label]

# 1) Identifiers
la_name = get_by_variable(capital, "LA_Name").astype(str).str.strip()
year    = get_by_variable(capital, "PeriodCode").astype(str).str.strip()

# 2) Keep ONLY measure columns (drop identifiers before aggregating)
id_vars = {"LA_Name","PeriodCode"}
last_level = -1  # last level holds the raw variable names
measure_mask = ~capital.columns.get_level_values(last_level).isin(id_vars)
cap_meas = capital.loc[:, measure_mask]

# 3) Collapse columns by category × subcategory (fast and vectorized)
#    -> result has columns indexed by (category, subcategory)
cap_by_cat = cap_meas.groupby(axis=1, level=["category","subcategory"]).sum()

# 4) Add row identifiers, then aggregate by borough × year
cap_by_cat = cap_by_cat.assign(LA_Name=la_name.values, year=year.values)

cap_agg = (
    cap_by_cat
    .groupby(["LA_Name","year"], as_index=False)
    .sum(numeric_only=True)
    # at this point columns are the (category, subcategory) MultiIndex, plus the two group keys if you keep them
)

# Optional: keep a tidy “wide” table indexed by borough × year, with (category, subcategory) columns
cap_wide = (
    cap_by_cat
    .groupby(["LA_Name","year"])
    .sum(numeric_only=True)
)


In [ ]:
import re
import numpy as np

def get_by_last_level_any(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    """Return first matching column by LAST column level (or flat)."""
    if isinstance(df.columns, pd.MultiIndex):
        last = df.columns.get_level_values(-1)
        for name in candidates:
            m = (last == name)
            if m.any():
                s = df.loc[:, m]
                return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s
        raise KeyError(f"None of {candidates} found. Last-level examples: {list(pd.unique(last))[:25]}")
    else:
        for name in candidates:
            if name in df.columns:
                return df[name]
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)[:25]}")

def is_total_var(code: str, desc: str | None = None) -> bool:
    """Heuristics for explicit totals in revenue & capital."""
    c = str(code).lower() if code is not None else ""
    d = str(desc).lower() if desc is not None else ""

    # Revenue style: ... _tot _ ...
    if re.search(r"(^|[_\-])tot([_\-]|$)", c):
        return True
    # Capital style: exptot..., rectot..., capsum...
    if c.startswith(("exptot", "rectot", "capsum")):
        return True
    # Descriptions that literally say total/grand total
    if "total" in d or "grand total" in d:
        return True
    return False

def select_total_columns(df: pd.DataFrame) -> pd.Index:
    """Return column index of explicit total variables (by code and description)."""
    assert isinstance(df.columns, pd.MultiIndex), "Expected MultiIndex columns."
    lvl_names = df.columns.names
    lvl_var = lvl_names[-1]
    lvl_desc = "description" if "description" in lvl_names else lvl_names[-2]

    codes = df.columns.get_level_values(lvl_var)
    descs = df.columns.get_level_values(lvl_desc)
    mask = [is_total_var(c, d) for c, d in zip(codes, descs)]
    return df.columns[pd.Index(mask)]


In [ ]:
# Borough names & year labels
rev_la   = get_by_last_level_any(rev, ["LA_name","LA_Name"]).astype(str).str.strip()
rev_year = get_by_last_level_any(rev, ["year_ending","Year_Ending"]).astype(str).str.slice(0,4)

cap_la   = get_by_last_level_any(capital, ["LA_name","LA_Name"]).astype(str).str.strip()
cap_per  = get_by_last_level_any(capital, ["PeriodCode","periodcode"]).astype(str)
cap_year = cap_per.str.slice(0,4)

# Aggregate row = "London"
rev_london_agg = rev[rev_la.eq("London")]
cap_london_agg = capital[cap_la.eq("London")]

# Borough rows = everything else
rev_boroughs = rev[~rev_la.eq("London")]
cap_boroughs = capital[~cap_la.eq("London")]



In [ ]:
rev_tot_cols = select_total_columns(rev)
cap_tot_cols = select_total_columns(capital)

print("Revenue total columns found:", len(rev_tot_cols))
print("Capital total columns found:", len(cap_tot_cols))


In [ ]:
# ----- REVENUE -----
# get year as YYYY for grouping
rev_y = rev_year.str.slice(0,4)

# Borough sums by year
rev_boro_sum = (
    rev_boroughs.loc[:, rev_tot_cols]
    .assign(__year=rev_y.loc[rev_boroughs.index].values)
    .groupby("__year")
    .sum(numeric_only=True)
)

# London aggregate by year (there should be one row per year)
rev_london_by_year = (
    rev_london_agg.loc[:, rev_tot_cols]
    .assign(__year=rev_y.loc[rev_london_agg.index].values)
    .groupby("__year")
    .sum(numeric_only=True)
)

# Align and compute differences
rev_cmp = rev_boro_sum.reindex_like(rev_london_by_year)
rev_diff = rev_boro_sum - rev_london_by_year
rev_max_abs_diff = rev_diff.abs().max(axis=1).rename("max_abs_diff")

print("Revenue: max absolute diff per year (borough sum vs London aggregate):")
display(rev_max_abs_diff)

# Show biggest mismatches
rev_top_mismatch = (rev_diff.stack().abs()
                    .rename("abs_diff")
                    .sort_values(ascending=False)
                    .groupby(level=0)
                    .head(10))
print("Revenue: top mismatches by year/column:")
display(rev_top_mismatch.head(20))

# ----- CAPITAL -----
# Borough sums by year
cap_boro_sum = (
    cap_boroughs.loc[:, cap_tot_cols]
    .assign(__year=cap_year.loc[cap_boroughs.index].values)
    .groupby("__year")
    .sum(numeric_only=True)
)

cap_london_by_year = (
    cap_london_agg.loc[:, cap_tot_cols]
    .assign(__year=cap_year.loc[cap_london_agg.index].values)
    .groupby("__year")
    .sum(numeric_only=True)
)

cap_diff = cap_boro_sum - cap_london_by_year
cap_max_abs_diff = cap_diff.abs().max(axis=1).rename("max_abs_diff")

print("Capital: max absolute diff per year (borough sum vs London aggregate):")
display(cap_max_abs_diff)

cap_top_mismatch = (cap_diff.stack().abs()
                    .rename("abs_diff")
                    .sort_values(ascending=False)
                    .groupby(level=0)
                    .head(10))
print("Capital: top mismatches by year/column:")
display(cap_top_mismatch.head(20))


In [ ]:
import numpy as np
import pandas as pd

# ---------- REVENUE ----------
# Borough sums by year (you already computed these earlier)
#   rev_boro_sum: DataFrame indexed by "__year", columns = MultiIndex (category, subcategory, ...maybe more)
#   rev_london_by_year: same shape

# Align columns/years first
rev_boro_sum = rev_boro_sum.reindex_like(rev_london_by_year)

# Differences
rev_diff = rev_boro_sum - rev_london_by_year

# 1) Per-year max absolute diff (good quick QA)
rev_max_abs_diff = rev_diff.abs().max(axis=1).rename("max_abs_diff")
print("Revenue: max absolute diff per year (borough sum vs London aggregate):")
display(rev_max_abs_diff)

# 2) Top mismatches per year (stack ALL column levels so we get a Series)
rev_diff_series = (
    rev_diff
    .stack(list(rev_diff.columns.names))   # <-- key change: stack every remaining col level
    .abs()
    .rename("abs_diff")
)

# largest mismatches per year (level=0 should be "__year")
rev_top_mismatch = (
    rev_diff_series
    .sort_values(ascending=False)
    .groupby(level=0)
    .head(10)
)
print("Revenue: top mismatches by year/column:")
display(rev_top_mismatch.head(20))

# Optional: flag years that pass a tolerance (e.g., ±£1)
TOL = 1.0
rev_pass = (rev_max_abs_diff <= TOL)
print("Revenue: years where borough sum == London aggregate within tolerance:", list(rev_pass[rev_pass].index))


# ---------- CAPITAL ----------
# Same pattern
cap_boro_sum = cap_boro_sum.reindex_like(cap_london_by_year)
cap_diff = cap_boro_sum - cap_london_by_year

cap_max_abs_diff = cap_diff.abs().max(axis=1).rename("max_abs_diff")
print("Capital: max absolute diff per year (borough sum vs London aggregate):")
display(cap_max_abs_diff)

cap_diff_series = (
    cap_diff
    .stack(list(cap_diff.columns.names))
    .abs()
    .rename("abs_diff")
)

cap_top_mismatch = (
    cap_diff_series
    .sort_values(ascending=False)
    .groupby(level=0)
    .head(10)
)
print("Capital: top mismatches by year/column:")
display(cap_top_mismatch.head(20))

cap_pass = (cap_max_abs_diff <= TOL)
print("Capital: years where borough sum == London aggregate within tolerance:", list(cap_pass[cap_pass].index))
